In [0]:
import logging
from typing import Any
from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from assignment_internal import AssignmentInternal
from pyspark.sql.functions import (
    coalesce,
    concat_ws,
    current_timestamp,
    expr,
    from_utc_timestamp,
    lit,
    max,
    sha2,
    substring,
    to_date,
    when,
    lag
)


In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_assignment_qca(
    spark, catalog_landed, run_date, env_prefix
):
        date_filter = f"businessdate = {run_date!r}"
        cluster_columns = ["BusinessDate"]
        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter  # possible need change date

        portfolio_selected_fields = [
            "PortfolioName",
            "Prefix",
            "portfolio_id_qca",
        ]

        curated_zone = "`p-cdh`.primary"
        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )
        # ================== dim_customer_static ==================

        dim_customer_static_extra_filters = None

        if dim_customer_static_extra_filters is not None:
            dim_customer_static_extra_filters.append(date_filter)
            dim_customer_static_final_filters = " AND ".join(
                dim_customer_static_extra_filters
            )
        else:
            dim_customer_static_final_filters = date_filter

        dim_customer_static_selected_fields = [
            "cust_id",
            "cust_cif",
            "cust_afm",
            "businessdate",
        ]

        dim_customer_static = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_customer_static",
            dim_customer_static_final_filters,
            dim_customer_static_selected_fields,
        )

        # ================== at_customer_extra ==================

        at_customer_extra_extra_filters = None

        if at_customer_extra_extra_filters is not None:
            at_customer_extra_extra_filters.append(date_filter)
            at_customer_extra_final_filters = " AND ".join(
                at_customer_extra_extra_filters
            )
        else:
            at_customer_extra_final_filters = date_filter

        at_customer_extra_selected_fields = [
            "cust_code",
            "CUEX_GENERIC_TEXT_16",
            "CUEX_GENERIC_TEXT_20",
        ]

        at_customer_extra = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_at_customer_extra",
            at_customer_extra_final_filters,
            at_customer_extra_selected_fields,
        )

        # ================== case_assignment ==================

        assignment_internal = AssignmentInternal(spark, catalog_landed, run_date, env_prefix)
        case_assignment = assignment_internal.loadCuratedAssignmentQca()

        # Rename columns to match what the rest of the notebook expects (lowercase)
        case_assignment = (
            case_assignment
            .withColumnRenamed("CASE_CODE", "case_code")
            .withColumnRenamed("CASE_ACCOUNT_FOR_SEARCH", "case_account_for_search")
            .withColumnRenamed("CUST_CODE", "cust_code")
            .withColumnRenamed("AssignmentChannel", "assignmentchannel")
            .withColumnRenamed("ExternalCompanyAssignment", "externalcompanyassignment")
            .withColumnRenamed("ExternalCompanyName", "externalcompanyname")
            .withColumnRenamed("AssignmentTeam", "assignmentteam")
            .withColumnRenamed("AssignmentAgentLoginName", "assignmentagentloginname")
            .withColumnRenamed("AssignmentAgent", "AssignmentAgent")
            .withColumnRenamed("AssignmentProcess", "AssignmentProcess")
            .withColumnRenamed("BusinessDate", "businessdate")
        )

        # Keep only the columns the notebook expects
        case_assignment = case_assignment.select(
            "case_code",
            "case_account_for_search",
            "cust_code",
            "assignmentchannel",
            "externalcompanyassignment",
            "externalcompanyname",
            "assignmentteam",
            "assignmentagentloginname",
            "AssignmentAgent",
            "AssignmentProcess",
            "businessdate",
        )

        # HashKeys (same as before)
        case_assignment_primary_key_fields = [
            case_assignment.case_code,
            lit("Channel Assignment"),
            lit("qca"),
        ]

        case_assignment = case_assignment.withColumn(
            "AssignmentId",
            sha2(concat_ws(";", *case_assignment_primary_key_fields), 256),
        ).withColumn(
            "AssignmentKey",
            concat_ws(
                ";",
                case_assignment.case_code,
                lit("Channel Assignment"),
            ),
        )

        # ================== fs_assignment_snap_leaf ==================

        fs_assignment_snap_leaf_extra_filters = None

        if fs_assignment_snap_leaf_extra_filters is not None:
            fs_assignment_snap_leaf_extra_filters.append(date_filter)
            fs_assignment_snap_leaf_final_filters = " AND ".join(
                fs_assignment_snap_leaf_extra_filters
            )
        else:
            fs_assignment_snap_leaf_final_filters = date_filter

        fs_assignment_snap_leaf_selected_fields = [
            "case_code",
            "cust_code",
            "cr_date",
            "bck_ass_id",
            "pack_id",
            "pacc_dateback",
            "pacc_code",
            "fsas_out_amt_ass",
            "fsas_dbt_amt",
            "ass_col_id_ass",
            "businessdate",
        ]

        fs_assignment_snap_leaf = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_fs_assignment_snap_leaf",
            fs_assignment_snap_leaf_final_filters,
            fs_assignment_snap_leaf_selected_fields,
        )

        # HashKeys
        fs_assignment_snap_leaf_primary_key_fields = [
            fs_assignment_snap_leaf.case_code,
            fs_assignment_snap_leaf.pacc_code,
            lit("qca"),
        ]

        fs_assignment_snap_leaf = fs_assignment_snap_leaf.withColumn(
            "AssignmentId",
            sha2(concat_ws(";", *fs_assignment_snap_leaf_primary_key_fields), 256),
        ).withColumn(
            "concatenated_cr_date",
            to_date(
                concat_ws(
                    "-",
                    substring(("cr_date"), 1, 4),
                    substring(("cr_date"), 5, 2),
                    substring(("cr_date"), 7, 2),
                ),
                "yyyy-MM-dd",
            ),
        ).distinct()
        
        # ================== fsad_cases_generic_open ==================

        fsad_cases_generic_open_extra_filters = None

        if fsad_cases_generic_open_extra_filters is not None:
            fsad_cases_generic_open_extra_filters.append(date_filter)
            fsad_cases_generic_open_final_filters = " AND ".join(fsad_cases_generic_open_extra_filters)
        else:
            fsad_cases_generic_open_final_filters = date_filter

        fsad_cases_generic_open_extra_selected_fields = [
            "case_code", 
            "cust_code",
            "debt_code",
            "casg_generic_bit_01",
            "casg_generic_bit_02",
            "casg_generic_nvarchar_01",
            "BusinessDate"
        ]

        fsad_cases_generic_open = load_landed_delta_table(
            spark,
            catalog_landed, 
            "qca_fsad_cases_generic_open",
            fsad_cases_generic_open_final_filters,
            fsad_cases_generic_open_extra_selected_fields,
        )
        
        fsad_cases_generic_open = fsad_cases_generic_open.withColumnRenamed(
            'cust_code', 'fsad_cases_generic_open_cust_code'
        )

        fsad_cases_generic_open = fsad_cases_generic_open.withColumnRenamed(
            'debt_code', 'fsad_cases_generic_open_debt_code'
        )
        
        fsad_cases_generic_open = fsad_cases_generic_open.withColumnRenamed(
            'case_code', 'fsad_cases_generic_open_case_code'
        )

        fsad_cases_generic_open = fsad_cases_generic_open.withColumnRenamed(
            'casg_generic_bit_01', 'ForcedOnboarding'
        )
        
        fsad_cases_generic_open = fsad_cases_generic_open.withColumnRenamed(
            'casg_generic_bit_02', 'HolisticFlag'
        )
        
        fsad_cases_generic_open = fsad_cases_generic_open.withColumnRenamed(
            'casg_generic_nvarchar_01', 'ForcedOnboardReason'
        )
        
        fsad_cases_generic_open = fsad_cases_generic_open.dropDuplicates(
            subset = ['fsad_cases_generic_open_cust_code', 'fsad_cases_generic_open_debt_code', 'fsad_cases_generic_open_case_code']
        )

        # ================== fs_cases_snap ==================

        fs_cases_snap_extra_filters = None

        if fs_cases_snap_extra_filters is not None:
            fs_cases_snap_extra_filters.append(date_filter)
            fs_cases_snap_final_filters = " AND ".join(fs_cases_snap_extra_filters)
        else:
            fs_cases_snap_final_filters = date_filter

        fs_cases_snap_selected_fields = [
            "extc_code_primary",
            "pacc_code",
            "case_code",
            "cust_code",
            "debt_code",
            "SPET_CODE",
            "que_code",
            "PACKH3_ID",
            "pack_code",
            "businessdate",
        ]

        fs_cases_snap = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_fs_cases_snap",
            fs_cases_snap_final_filters,
            fs_cases_snap_selected_fields,
        )

        fs_cases_snap = fs_cases_snap.withColumnRenamed(
            "case_code", "fs_cases_snap_case_code"
        )
        
        fs_cases_snap = fs_cases_snap.withColumnRenamed(
            "debt_code", "fs_cases_snap_debt_code"
        )

        fs_cases_snap = fs_cases_snap.withColumnRenamed(
            "que_code", "fs_cases_snap_que_code"
        )

        fs_cases_snap = fs_cases_snap.withColumnRenamed(
            "pack_code", "fs_cases_snap_pack_code"
            )

        # take distinct values
        fs_cases_snap_customers = fs_cases_snap.select(
            fs_cases_snap.cust_code
        ).distinct()

        # ================== dim_cases_static ==================

        dim_cases_static_extra_filters = None

        if dim_cases_static_extra_filters is not None:
            dim_cases_static_extra_filters.append(date_filter)
            dim_cases_static_final_filters = " AND ".join(
                dim_cases_static_extra_filters
            )
        else:
            dim_cases_static_final_filters = date_filter

        dim_cases_static_selected_fields = [
            "case_account_for_search",
            "case_code",
            "businessdate",
        ]

        dim_cases_static = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_cases_static",
            dim_cases_static_final_filters,
            dim_cases_static_selected_fields,
        )

        dim_cases_static = dim_cases_static.withColumnRenamed(
            "case_code", "dim_cases_static_case_code"
        )

        # ================== dim_external_company ==================

        dim_external_company_extra_filters = None

        if dim_external_company_extra_filters is not None:
            dim_external_company_extra_filters.append(date_filter)
            dim_external_company_final_filters = " AND ".join(
                dim_external_company_extra_filters
            )
        else:
            dim_external_company_final_filters = date_filter

        dim_external_company_selected_fields = ["extc_id", "EXTC_NAME"]

        dim_external_company = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_external_company",
            dim_external_company_final_filters,
            dim_external_company_selected_fields,
        )

        dim_external_company = dim_external_company.withColumnRenamed(
            "extc_id", "DEC_extc_id"
        ).withColumnRenamed("EXTC_NAME", "DEC_EXTC_NAME")

        # ================== dim_packet ==================

        dim_packet_extra_filters = None

        if dim_packet_extra_filters is not None:
            dim_packet_extra_filters.append(date_filter)
            dim_packet_final_filters = " AND ".join(dim_packet_extra_filters)
        else:
            dim_packet_final_filters = date_filter

        dim_packet_selected_fields = [
            "pack_name",
            "pack_assign_date",
            "pack_id",
            "businessdate",
        ]

        dim_packet = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_packet",
            dim_packet_final_filters,
            dim_packet_selected_fields,
        )

        # ================== dim_bucket ==================

        dim_bucket_extra_filters = None

        if dim_bucket_extra_filters is not None:
            dim_bucket_extra_filters.append(date_filter)
            dim_bucket_final_filters = " AND ".join(dim_bucket_extra_filters)
        else:
            dim_bucket_final_filters = date_filter

        dim_bucket_selected_fields = ["bck_desc", "bck_id", "businessdate"]

        dim_bucket = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_bucket",
            dim_bucket_final_filters,
            dim_bucket_selected_fields,
        )

        # ================== dim_collector ==================

        dim_collector_extra_filters = None

        if dim_collector_extra_filters is not None:
            dim_collector_extra_filters.append(date_filter)
            dim_collector_final_filters = " AND ".join(dim_collector_extra_filters)
        else:
            dim_collector_final_filters = date_filter

        dim_collector_selected_fields = [
            "col_id",
            "COL_FNAME",
            "COL_LNAME",
            "col_login",
        ]

        dim_collector = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_collector",
            dim_collector_final_filters,
            dim_collector_selected_fields,
        )

        dim_collector = dim_collector.withColumn(
            "Agent_Full_Name",
            coalesce(
                expr(
                    "NULLIF(concat_ws(' ', trim(COL_FNAME), trim(COL_LNAME)), ' ')"
                ),  # Concatenate if both are non-null and non-empty)
                expr(
                    "NULLIF(trim(COL_FNAME), '')"
                ),  # Take firstname if lastname is null or empty
                expr(
                    "NULLIF(trim(COL_LNAME), '')"
                ),  # Take lastname if firstname is null or empty
            ),
        )

        # ================== dim_queue ==================

        dim_queue_extra_filters = None

        if dim_queue_extra_filters is not None:
            dim_queue_extra_filters.append(date_filter)
            dim_queue_final_filters = " AND ".join(dim_queue_extra_filters)
        else:
            dim_queue_final_filters = date_filter

        dim_queue_selected_fields = ["que_desc", "que_id", "businessdate"]

        dim_queue = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_queue",
            dim_queue_final_filters,
            dim_queue_selected_fields,
        )

        # Added on 15/1
        # Author: V. Iurcovschi
        # ================== at_cases_queue =====================

        at_cases_queue_extra_filters = None

        if at_cases_queue_extra_filters is not None:
            at_cases_queue_extra_filters.append(date_filter)
            at_cases_queue_final_filters = " AND ".join(at_cases_queue_extra_filters)
        else:
            at_cases_queue_final_filters = date_filter

        at_cases_queue_selected_fields = [
            "case_code", 
            "caqu_date", 
            "caqu_que_desc", 
            "que_code",
            ]
        
        at_cases_queue = load_landed_delta_table(
            spark,
            catalog_landed, 
            "qualcoalpha_atcasequeuehistory",
            at_cases_queue_final_filters,
            at_cases_queue_selected_fields,
        )

        historical_at_cases_queue = spark.sql(f"""SELECT CASE_CODE, CAQU_DATE, CAQU_QUE_DESC, QUE_CODE FROM `{env_prefix}-cdh`.reference.atcasesloadqueue""")
        at_cases_queue = at_cases_queue.union(historical_at_cases_queue)

        window_spec = Window.partitionBy("case_code").orderBy(col("caqu_date").desc())
        at_cases_queue = (
            at_cases_queue
            .withColumn("caqu_start_date", col("caqu_date").cast("date"))
            .withColumn("caqu_end_date", coalesce(lag("caqu_date").over(window_spec), col("caqu_date")).cast("date")
            )
        )
        at_cases_queue = at_cases_queue.withColumnRenamed(
            "case_code","case_code_queue"
            )
        
        at_cases_queue = at_cases_queue.withColumnRenamed(
            "que_code", "caqu_que_code"
            )

        # ================== dim_stream_status ==================

        dim_stream_status_extra_filters = None

        if dim_stream_status_extra_filters is not None:
            dim_stream_status_extra_filters.append(date_filter)
            dim_stream_status_final_filters = " AND ".join(
                dim_stream_status_extra_filters
            )
        else:
            dim_stream_status_final_filters = date_filter

        dim_stream_status_selected_fields = ["strm_stat_desc", "strm_stat_id"]

        dim_stream_status = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_stream_status",
            dim_stream_status_final_filters,
            dim_stream_status_selected_fields,
        )

        # ================== dim_stream_type ==================

        dim_stream_type_extra_filters = None

        if dim_stream_type_extra_filters is not None:
            dim_stream_type_extra_filters.append(date_filter)
            dim_stream_type_final_filters = " AND ".join(dim_stream_type_extra_filters)
        else:
            dim_stream_type_final_filters = date_filter

        dim_stream_type_selected_fields = ["strm_desc", "strm_id"]

        dim_stream_type = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_stream_type",
            dim_stream_type_final_filters,
            dim_stream_type_selected_fields,
        )

        # ================== dim_packet_hier3 ==================

        dim_packet_hier3_extra_filters = None

        if dim_packet_hier3_extra_filters is not None:
            dim_packet_hier3_extra_filters.append(date_filter)
            dim_packet_hier3_final_filters = " AND ".join(dim_packet_hier3_extra_filters)
        else:
            dim_packet_hier3_final_filters = date_filter

        dim_packet_hier3_selected_fields = [
            "PACKH3_DESC",
            "PACKH3_ID"
        ]

        dim_packet_hier3 = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_packet_hier3",
            dim_packet_hier3_final_filters,
            dim_packet_hier3_selected_fields,
        )

        # ================== fs_case_stream_assignments_snap_leaf ==================

        fs_case_stream_assignments_snap_leaf_extra_filters = None

        if fs_case_stream_assignments_snap_leaf_extra_filters is not None:
            fs_case_stream_assignments_snap_leaf_extra_filters.append(date_filter)
            fs_case_stream_assignments_snap_leaf_final_filters = " AND ".join(
                fs_case_stream_assignments_snap_leaf_extra_filters
            )
        else:
            fs_case_stream_assignments_snap_leaf_final_filters = date_filter

        fs_case_stream_assignments_snap_leaf_selected_fields = [
            "case_code",
            "cust_code",
            "cr_date",
            "extc_code",
            "pacc_bucket",
            "pack_code",
            "pacc_dateback",
            "pack_assign_date",
            "pacc_code",
            "que_code",
            "strm_id",
            "strm_sta_id",
            "strm_sta_ass_id",
            "pacc_amount",
            "pacc_debt_amount",
            "businessdate",
        ]

        fs_case_stream_assignments_snap_leaf = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_fs_case_stream_assignments_snap_leaf",
            fs_case_stream_assignments_snap_leaf_final_filters,
            fs_case_stream_assignments_snap_leaf_selected_fields,
        )

        fs_case_stream_assignments_snap_leaf = fs_case_stream_assignments_snap_leaf.distinct()

        # HashKeys
        # fs_case_stream_assignments_snap_leaf_primary_key_fields = [
        #    fs_case_stream_assignments_snap_leaf.case_code,
        #    lit("qca"),
        # ]

        fs_case_stream_assignments_snap_leaf = (
            fs_case_stream_assignments_snap_leaf.withColumn(
                "concatenated_cr_date",
                to_date(
                    concat_ws(
                        "-",
                        substring(("cr_date"), 1, 4),
                        substring(("cr_date"), 5, 2),
                        substring(("cr_date"), 7, 2),
                    ),
                    "yyyy-MM-dd",
                ),
            )
        ).distinct()

        # ================== fs_customer_stream_assignments_snap_leaf ==================

        fs_customer_stream_assignments_snap_leaf_extra_filters = None

        if fs_customer_stream_assignments_snap_leaf_extra_filters is not None:
            fs_customer_stream_assignments_snap_leaf_extra_filters.append(date_filter)
            fs_customer_stream_assignments_snap_leaf_final_filters = " AND ".join(
                fs_customer_stream_assignments_snap_leaf_extra_filters
            )
        else:
            fs_customer_stream_assignments_snap_leaf_final_filters = date_filter

        fs_customer_stream_assignments_snap_leaf_selected_fields = [
            "case_code",
            "cust_code",
            "cr_date",
            "extc_code",
            "pack_code",
            "pacc_code",
            "pacc_outstanding_amt_lcy",
            "businessdate",
        ]

        fs_customer_stream_assignments_snap_leaf = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_fs_customer_stream_assignments_snap_leaf",
            fs_customer_stream_assignments_snap_leaf_final_filters,
            fs_customer_stream_assignments_snap_leaf_selected_fields,
        )

        # HashKeys
        fs_customer_stream_assignments_snap_leaf_primary_key_fields = [
            fs_customer_stream_assignments_snap_leaf.case_code,
            lit("qca"),
        ]

        fs_customer_stream_assignments_snap_leaf = (
            fs_customer_stream_assignments_snap_leaf.withColumn(
                "AssignmentId",
                sha2(
                    concat_ws(
                        ";",
                        *fs_customer_stream_assignments_snap_leaf_primary_key_fields,
                    ),
                    256,
                ),
            ).withColumn(
                "concatenated_cr_date",
                to_date(
                    concat_ws(
                        "-",
                        substring(("cr_date"), 1, 4),
                        substring(("cr_date"), 5, 2),
                        substring(("cr_date"), 7, 2),
                    ),
                    "yyyy-MM-dd",
                ),
            )
        )

        # Core Retail (FS_ASSIGNMENT_SNAP_LEAF)
        # Conditions
        fs_cases_snap_fs_assignment_snap_leaf_join_condition = (
            fs_cases_snap["pacc_code"] == fs_assignment_snap_leaf["pacc_code"]
        ) & (fs_assignment_snap_leaf["pacc_dateback"].isNull())

        dim_cases_static_fs_assignment_snap_leaf_join_condition = (
            dim_cases_static["dim_cases_static_case_code"]
            == fs_assignment_snap_leaf["case_code"]
        )
        dim_external_company_fs_cases_snap_join_condition = (
            dim_external_company["DEC_extc_id"] == fs_cases_snap["extc_code_primary"]
        )
        dim_packet_fs_assignment_snap_leaf_join_condition = (
            dim_packet["pack_id"] == fs_assignment_snap_leaf["pack_id"]
        )

        dim_bucket_fs_assignment_snap_leaf_join_condition = (
            dim_bucket["bck_id"] == fs_assignment_snap_leaf["bck_ass_id"]
        )

        dim_collector_fs_assignment_snap_leaf_join_condition = (
            dim_collector["col_id"] == fs_assignment_snap_leaf["ass_col_id_ass"]
        )
        dim_queue_fs_cases_snap_join_condition = (
            dim_queue["que_id"] == fs_cases_snap["fs_cases_snap_que_code"]
        )

        fs_cases_snap_alias = fs_cases_snap.alias("fs_cases_snap")
        fs_cases_snap_alias = fs_cases_snap_alias.withColumnRenamed(
            "fs_cases_snap_case_code", "fs_cases_snap_alias_case_code"
        )
        fs_cases_snap_alias = fs_cases_snap_alias.select(
            fs_cases_snap_alias.fs_cases_snap_alias_case_code
        )

        dim_cases_static_fs_cases_snap_join_condition = (
            dim_cases_static["dim_cases_static_case_code"]
            == fs_cases_snap_alias["fs_cases_snap_alias_case_code"]
        )

        fs_cases_snap_portfolio_join_condition = (
            fs_cases_snap["SPET_CODE"] == portfolio["portfolio_id_qca"]
        )

        dim_customer_static_fs_assignment_snap_leaf_join_condition = (
            dim_customer_static["cust_id"] == fs_assignment_snap_leaf["cust_code"]
        )

        dim_packet_hier3_fs_assignment_snap_leaf_join_condition = (
            dim_packet_hier3["PACKH3_ID"] == fs_cases_snap["PACKH3_ID"]
        )
        
        fsad_cases_generic_open_join_condition = (
          fsad_cases_generic_open['fsad_cases_generic_open_case_code'] == fs_cases_snap["fs_cases_snap_case_code"]
          ) & (fsad_cases_generic_open['fsad_cases_generic_open_cust_code'] == fs_cases_snap["cust_code"]
          ) & (fsad_cases_generic_open['fsad_cases_generic_open_debt_code'] == fs_cases_snap["fs_cases_snap_debt_code"]) 

        # Added on 15/1
        # Author: V. Iurcovschi
        at_cases_queue_open_join_condition = (
            fs_assignment_snap_leaf["case_code"] == at_cases_queue["case_code_queue"]
            ) & (fs_assignment_snap_leaf["BusinessDate"] >= at_cases_queue["caqu_start_date"]
            ) & (fs_assignment_snap_leaf["BusinessDate"] < at_cases_queue["caqu_end_date"])

        at_customer_extra_fs_assignment_snap_leaf_join_condition = (
            at_customer_extra["cust_code"] == fs_assignment_snap_leaf["cust_code"]
        )
        

        # Join
        join_fs_assignment_snap_leaf_df = (
            fs_cases_snap.join(
                fs_assignment_snap_leaf,
                fs_cases_snap_fs_assignment_snap_leaf_join_condition,
                "inner",
            )
            .join(
              at_cases_queue,
              at_cases_queue_open_join_condition,
              "left",
            )
            .join(
                fsad_cases_generic_open,
                fsad_cases_generic_open_join_condition,
                "left",
            )
            .join(
                dim_cases_static,
                dim_cases_static_fs_assignment_snap_leaf_join_condition,
                "left",
            )
            .join(
                dim_external_company,
                dim_external_company_fs_cases_snap_join_condition,
                "left",
            )
            .join(
                dim_packet,
                dim_packet_fs_assignment_snap_leaf_join_condition,
                "left",
            )
            .join(
                dim_bucket,
                dim_bucket_fs_assignment_snap_leaf_join_condition,
                "left",
            )
            .join(
                dim_collector,
                dim_collector_fs_assignment_snap_leaf_join_condition,
                "left",
            )
            .join(
                dim_queue,
                dim_queue_fs_cases_snap_join_condition,
                "left",
            )
            .join(
                fs_cases_snap_alias,
                dim_cases_static_fs_cases_snap_join_condition,
                "inner",
            )
            .join(
                portfolio,
                fs_cases_snap_portfolio_join_condition,
                "left",
            )
            .join(
                dim_customer_static,
                dim_customer_static_fs_assignment_snap_leaf_join_condition,
                "inner",
            )
            .join(
                dim_packet_hier3,
                dim_packet_hier3_fs_assignment_snap_leaf_join_condition,
                "left",
            )
            .join(
                at_customer_extra,
                at_customer_extra_fs_assignment_snap_leaf_join_condition,
                "left",
            )        
        ).distinct()

        # HashKeys
        join_fs_assignment_snap_leaf_df_leaf_primary_key_fields = [
            fs_assignment_snap_leaf.case_code,
            fs_assignment_snap_leaf.pacc_code,
            lit("Collection Assignment"),
            lit("qca"),
        ]

        join_fs_assignment_snap_leaf_df = join_fs_assignment_snap_leaf_df.withColumn(
            "AssignmentId",
            sha2(
                concat_ws(
                    ";", *join_fs_assignment_snap_leaf_df_leaf_primary_key_fields
                ),
                256,
            ),
        ).withColumn(
            "AssignmentKey",
            concat_ws(
                ";",
                fs_assignment_snap_leaf.case_code,
                fs_assignment_snap_leaf.pacc_code,
                lit("Collection Assignment"),
            ),
        )

        # Core Retail (FS_CASE_STREAM_ASSIGNMENTS_SNAP_LEAF)
        # Rename the columns in each DataFrame to avoid ambiguity
        fs_case_stream_assignments_snap_leaf = (
            fs_case_stream_assignments_snap_leaf.withColumnRenamed(
                "case_code", "fs_case_stream_assignments_snap_leaf_case_code"
            ).withColumnRenamed(
                "strm_id", "fs_case_stream_assignments_snap_leaf_strm_id"
            )
        )

        dim_stream_type = dim_stream_type.withColumnRenamed(
            "strm_id", "dim_stream_type_strm_id"
        )

        # Conditions
        fs_case_stream_assignments_snap_leaf_fs_cases_snap_join_condition = (
            fs_case_stream_assignments_snap_leaf[
                "fs_case_stream_assignments_snap_leaf_case_code"
            ]
            == fs_cases_snap["fs_cases_snap_case_code"]
        )

        dim_cases_static_fs_case_stream_assignments_snap_leaf_join_condition = (
            dim_cases_static["dim_cases_static_case_code"]
            == fs_case_stream_assignments_snap_leaf[
                "fs_case_stream_assignments_snap_leaf_case_code"
            ]
        )

        dim_external_company_fs_case_stream_assignments_snap_leaf_join_condition = (
            dim_external_company["DEC_extc_id"]
            == fs_case_stream_assignments_snap_leaf["extc_code"]
        )

        dim_bucket_fs_case_stream_assignments_snap_leaf_join_condition = (
            dim_bucket["bck_id"] == fs_case_stream_assignments_snap_leaf["pacc_bucket"]
        )

        dim_packet_fs_case_stream_assignments_snap_leaf_join_condition = (
            dim_packet["pack_id"] == fs_case_stream_assignments_snap_leaf["pack_code"]
        )

        dim_stream_type_fs_case_stream_assignments_snap_leaf_join_condition = (
            dim_stream_type["dim_stream_type_strm_id"]
            == fs_case_stream_assignments_snap_leaf[
                "fs_case_stream_assignments_snap_leaf_strm_id"
            ]
        )

        fs_case_stream_assignments_snap_leaf2 = fs_case_stream_assignments_snap_leaf

        # subjoin1
        subjoin1_condition = (
            fs_case_stream_assignments_snap_leaf2[
                "fs_case_stream_assignments_snap_leaf_case_code"
            ]
            == fs_cases_snap["fs_cases_snap_case_code"]
        )

        subjoin1 = (
            fs_case_stream_assignments_snap_leaf2.join(
                fs_cases_snap, subjoin1_condition, "inner"
            )
            .groupBy(
                fs_case_stream_assignments_snap_leaf2[
                    "fs_case_stream_assignments_snap_leaf_case_code"
                ],
                fs_case_stream_assignments_snap_leaf2[
                    "fs_case_stream_assignments_snap_leaf_strm_id"
                ],
            )
            .agg(
                max(fs_case_stream_assignments_snap_leaf2.pacc_code).alias("pacc_code")
            )
        )

        # Rename columns to avoid complexity
        subjoin1 = subjoin1.withColumnRenamed(
            "fs_case_stream_assignments_snap_leaf_case_code", "subjoin1_case_code"
        ).withColumnRenamed(
            "fs_case_stream_assignments_snap_leaf_strm_id", "subjoin1_strm_id"
        )

        # end of subjoin1

        subjoin_fs_case_stream_assignments_snap_leaf_join_condition = (
            (subjoin1["pacc_code"] == fs_case_stream_assignments_snap_leaf["pacc_code"])
            & (
                subjoin1["subjoin1_case_code"]
                == fs_case_stream_assignments_snap_leaf[
                    "fs_case_stream_assignments_snap_leaf_case_code"
                ]
            )
            & (
                subjoin1["subjoin1_strm_id"]
                == fs_case_stream_assignments_snap_leaf[
                    "fs_case_stream_assignments_snap_leaf_strm_id"
                ]
            )
        )

        dim_queue_fs_case_stream_assignments_snap_leaf_join_condition2 = (
            dim_queue["que_id"] == fs_case_stream_assignments_snap_leaf["que_code"]
        )

        # alias
        dim_stream_status_alias1 = dim_stream_status
        dim_stream_status_alias2 = dim_stream_status

        # Rename the column in DataFrame to avoid ambiguity
        dim_stream_status_alias2 = dim_stream_status_alias2.withColumnRenamed(
            "strm_stat_id", "dim_stream_status_alias2_strm_stat_id"
        ).withColumnRenamed("strm_stat_desc", "dim_stream_status_alias2_strm_stat_desc")

        dim_stream_status_fs_case_stream_assignments_snap_leaf_join_condition = (
            dim_stream_status_alias1["strm_stat_id"]
            == fs_case_stream_assignments_snap_leaf["strm_sta_id"]
        )

        dim_stream_status_fs_case_stream_assignments_snap_leaf_join_condition2 = (
            dim_stream_status_alias2["dim_stream_status_alias2_strm_stat_id"]
            == fs_case_stream_assignments_snap_leaf["strm_sta_ass_id"]
        )

        fs_cases_snap_portfolio_join_condition = (
            fs_cases_snap["SPET_CODE"] == portfolio["portfolio_id_qca"]
        )

        dim_customer_static_fs_case_stream_assignments_snap_leaf_join_condition = (
            dim_customer_static["cust_id"]
            == fs_case_stream_assignments_snap_leaf["cust_code"]
        )

        dim_packet_hier3_fs_assignment_snap_leaf_join_condition = (
            dim_packet_hier3["PACKH3_ID"] == fs_cases_snap["PACKH3_ID"]
        )
        
        fsad_cases_generic_open_join_condition = (
          fsad_cases_generic_open['fsad_cases_generic_open_case_code'] == fs_cases_snap["fs_cases_snap_case_code"]
          ) & (fsad_cases_generic_open['fsad_cases_generic_open_cust_code'] == fs_cases_snap["cust_code"]
          ) & (fsad_cases_generic_open['fsad_cases_generic_open_debt_code'] == fs_cases_snap["fs_cases_snap_debt_code"]) 

        # Added on 15/1
        # Author: V. Iurcovschi
        stream_assignments_at_cases_queue_open_join_condition = (
            fs_cases_snap["fs_cases_snap_case_code"] == at_cases_queue["case_code_queue"]
            ) & (fs_cases_snap["BusinessDate"] >= at_cases_queue["caqu_start_date"]
            ) & (fs_cases_snap["BusinessDate"] < at_cases_queue["caqu_end_date"])

        # join
        join_fs_case_stream_assignments_snap_leaf_df = (
            fs_case_stream_assignments_snap_leaf.join(
                fs_cases_snap,
                fs_case_stream_assignments_snap_leaf_fs_cases_snap_join_condition,
                "inner",
            )
            .join(
               fsad_cases_generic_open,
               fsad_cases_generic_open_join_condition,
               "left" 
            )
            .join(
                dim_cases_static,
                dim_cases_static_fs_case_stream_assignments_snap_leaf_join_condition,
                "left",
            )
            .join(
                dim_external_company,
                dim_external_company_fs_case_stream_assignments_snap_leaf_join_condition,
                "left",
            )
            .join(
                dim_packet,
                dim_packet_fs_case_stream_assignments_snap_leaf_join_condition,
                "left",
            )
            .join(
                dim_bucket,
                dim_bucket_fs_case_stream_assignments_snap_leaf_join_condition,
                "left",
            )
            .join(
                dim_stream_type,
                dim_stream_type_fs_case_stream_assignments_snap_leaf_join_condition,
                "left",
            )
            .join(
                subjoin1,
                subjoin_fs_case_stream_assignments_snap_leaf_join_condition,
                "inner",
            )
            .join(
                dim_stream_status_alias1,
                dim_stream_status_fs_case_stream_assignments_snap_leaf_join_condition,
                "left",
            )
            .join(
                dim_stream_status_alias2,
                dim_stream_status_fs_case_stream_assignments_snap_leaf_join_condition2,
                "left",
            )
            .join(
                dim_queue,
                dim_queue_fs_case_stream_assignments_snap_leaf_join_condition2,
                "left",
            )
            .join(
                at_cases_queue,
                stream_assignments_at_cases_queue_open_join_condition,
                "left",
            )
            .join(
                portfolio,
                fs_cases_snap_portfolio_join_condition,
                "left",
            )
            .join(
                dim_customer_static,
                dim_customer_static_fs_case_stream_assignments_snap_leaf_join_condition,
                "inner",
            )
            .join(
                dim_packet_hier3,
                dim_packet_hier3_fs_assignment_snap_leaf_join_condition,
                "left"
            )
        ).distinct()

        # HashKeys
        join_fs_case_stream_assignments_snap_leaf_df_primary_key_fields = [
            fs_case_stream_assignments_snap_leaf.fs_case_stream_assignments_snap_leaf_case_code,
            fs_case_stream_assignments_snap_leaf.pacc_code,
            lit("Legal Action Assignment"),
            lit("qca"),
        ]

        join_fs_case_stream_assignments_snap_leaf_df = join_fs_case_stream_assignments_snap_leaf_df.withColumn(
            "AssignmentId",
            sha2(
                concat_ws(
                    ";",
                    *join_fs_case_stream_assignments_snap_leaf_df_primary_key_fields,
                ),
                256,
            ),
        ).withColumn(
            "AssignmentKey",
            concat_ws(
                ";",
                fs_case_stream_assignments_snap_leaf.fs_case_stream_assignments_snap_leaf_case_code,
                fs_case_stream_assignments_snap_leaf.pacc_code,
                lit("Legal Action Assignment"),
            ),
        )

        # Core Retail (FS_CUSTOMER_STREAM_ASSIGNMENTS_SNAP_LEAF)

        # Conditions
        fs_customer_stream_assignments_snap_leaf_fs_cases_snap_join_condition = (
            fs_customer_stream_assignments_snap_leaf["cust_code"]
            == fs_cases_snap_customers["cust_code"]
        )

        dim_external_company_fs_customer_stream_assignments_snap_leaf_join_condition = (
            dim_external_company["DEC_extc_id"]
            == fs_customer_stream_assignments_snap_leaf["extc_code"]
        )
        dim_packet_fs_customer_stream_assignments_snap_leaf_join_condition = (
            dim_packet["pack_id"]
            == fs_customer_stream_assignments_snap_leaf["pack_code"]
        )

        fs_customer_stream_assignments_snap_leaf2 = (
            fs_customer_stream_assignments_snap_leaf
        )

        # l2 table
        l2 = fs_customer_stream_assignments_snap_leaf2.groupBy("cust_code").agg(
            max("pacc_code").alias("PAC")
        )

        # Rename the column in DataFrame to avoid ambiguity
        l2 = l2.withColumnRenamed("cust_code", "l2_cust_code")

        dim_customer_static_fs_cases_snap_customers_join_condition = (
            dim_customer_static["cust_id"] == fs_cases_snap_customers["cust_code"]
        )

        dim_customer_static_portfolio_join_condition = (
            substring(dim_customer_static["cust_cif"], 1, 3) == portfolio["Prefix"]
        )

        l2_fs_customer_stream_assignments_snap_leaf_join_condition = (
            l2["l2_cust_code"] == fs_customer_stream_assignments_snap_leaf["cust_code"]
        ) & (l2["PAC"] == fs_customer_stream_assignments_snap_leaf["pacc_code"])

        join_fs_customer_stream_assignments_snap_leaf_df = (
            fs_customer_stream_assignments_snap_leaf.join(
                fs_cases_snap_customers,
                fs_customer_stream_assignments_snap_leaf_fs_cases_snap_join_condition,
                "inner",
            )
            .join(
                dim_packet,
                dim_packet_fs_customer_stream_assignments_snap_leaf_join_condition,
                "left",
            )
            .join(
                l2,
                l2_fs_customer_stream_assignments_snap_leaf_join_condition,
                "inner",
            )
            .join(
                dim_customer_static,
                dim_customer_static_fs_cases_snap_customers_join_condition,
                "left",
            )
            .join(
                portfolio,
                dim_customer_static_portfolio_join_condition,
                "left",
            )
            .join(
                dim_external_company,
                dim_external_company_fs_customer_stream_assignments_snap_leaf_join_condition,
                "left",
            )
        )

        # .join(
        #        dim_cases_static,
        #        dim_cases_static_fs_cases_snap_join_condition,
        #        "left",
        #    )
        #    .join(
        #        dim_external_company,
        #        dim_external_company_fs_customer_stream_assignments_snap_leaf_join_condition,
        #        "left",
        #    )
        #    .join(
        #        dim_packet,
        #        dim_packet_fs_customer_stream_assignments_snap_leaf_join_condition,
        #        "left",
        #    )

        # HashKeys
        join_fs_customer_stream_assignments_snap_leaf_df_primary_key_fields = [
            fs_customer_stream_assignments_snap_leaf["cust_code"],
            fs_customer_stream_assignments_snap_leaf["pacc_code"],
            lit("Real Estate Research Assignment"),
            lit("qca"),
        ]

        join_fs_customer_stream_assignments_snap_leaf_df = join_fs_customer_stream_assignments_snap_leaf_df.withColumn(
            "AssignmentId",
            sha2(
                concat_ws(
                    ";",
                    *join_fs_customer_stream_assignments_snap_leaf_df_primary_key_fields,
                ),
                256,
            ),
        ).withColumn(
            "AssignmentKey",
            concat_ws(
                ";",
                fs_customer_stream_assignments_snap_leaf["cust_code"],
                fs_customer_stream_assignments_snap_leaf["pacc_code"],
                lit("Real Estate Research Assignment"),
            ),
        )

        # Core Retail (case_assignment)

        # Conditions

        fs_cases_snap_portfolio_join_condition = (
            fs_cases_snap["SPET_CODE"] == portfolio["portfolio_id_qca"]
        )

        fs_cases_snap_case_assignment_join_condition = (
            case_assignment["case_code"] == fs_cases_snap["fs_cases_snap_case_code"]
        )

        dim_customer_static_fs_cases_snap_join_condition = (
            dim_customer_static["cust_id"] == fs_cases_snap["cust_code"]
        )

        dim_collector_case_assignment_join_condition = (
            dim_collector["COL_ID"] == case_assignment["AssignmentAgent"]
        )

        dim_packet_case_assignment_join_condition = (
            dim_packet["pack_id"] == fs_cases_snap["fs_cases_snap_pack_code"]
        )

        dim_packet_hier3_join_condition = (
            dim_packet_hier3["PACKH3_ID"] == fs_cases_snap["PACKH3_ID"]
        )

        dim_queue_join_condition = (
            dim_queue["que_id"] == fs_cases_snap["fs_cases_snap_que_code"]
        )
        
        fsad_cases_generic_open_join_condition = (
          fsad_cases_generic_open['fsad_cases_generic_open_case_code'] == fs_cases_snap["fs_cases_snap_case_code"]
          ) & (fsad_cases_generic_open['fsad_cases_generic_open_cust_code'] == fs_cases_snap["cust_code"]
          ) & (fsad_cases_generic_open['fsad_cases_generic_open_debt_code'] == fs_cases_snap["fs_cases_snap_debt_code"]) 

        at_customer_extra_fs_cases_snap_join_condition = (
            at_customer_extra["cust_code"] == fs_cases_snap["cust_code"]
        )

        join_case_assignment_df = (
            case_assignment.join(
                fs_cases_snap,
                fs_cases_snap_case_assignment_join_condition,
                "inner",
            )
            .join(
                dim_customer_static,
                dim_customer_static_fs_cases_snap_join_condition,
                "inner",
            )
            .join(
                fsad_cases_generic_open,
                fsad_cases_generic_open_join_condition,
                "left",
            )
            .join(
                dim_collector,
                dim_collector_case_assignment_join_condition,
                "left",
            )
            .join(
                portfolio,
                fs_cases_snap_portfolio_join_condition,
                "left",
            )
            .join(
                dim_packet, 
                dim_packet_case_assignment_join_condition, 
                "left")
            .join(
                dim_packet_hier3,
                dim_packet_hier3_join_condition,
                "left"
            )
            .join(
                dim_queue,
                dim_queue_join_condition,
                "left",
            )
            .join(
                at_customer_extra,
                at_customer_extra_fs_cases_snap_join_condition,
                "left",
            )
        )

        join_case_assignment_df = (
            join_case_assignment_df.withColumn(
                "Team",
                when(
                    case_assignment["assignmentteam"] == '-1', None
                ).otherwise(  # If assignmentteam is -1, set to null
                    case_assignment["assignmentteam"]
                ),  # Else return assignmentteam value
            )
            .withColumn(
                "Channel",
                when(
                    case_assignment["assignmentchannel"] == '-1', None
                ).otherwise(  # If assignmentteam is -1, set to null
                    case_assignment["assignmentchannel"]
                ),  # Else return assignmentteam value
            )
            .withColumn(
                "AgentPartyKey",
                when(
                    case_assignment["AssignmentAgent"] == -1, None
                ).otherwise(  # If assignmentteam is -1, set to null
                    case_assignment["AssignmentAgent"]
                ),  # Else return assignmentteam value
            )
        )

        # .withColumn(
        #        "AgentFullName",
        #        when(
        #            (dim_collector["COL_FNAME"].isNull())
        #            | (dim_collector["COL_LNAME"].isNull())
        #            | (dim_collector["COL_FNAME"] == "")
        #            | (dim_collector["COL_LNAME"] == ""),
        #            None,
        #        ).otherwise(
        #            concat_ws(
        #                " ", dim_collector["COL_FNAME"], dim_collector["COL_LNAME"]
        #            )
        #        ),
        #    )

        # ------------ dfs ------------------

        # case_assignment_df
        case_assignment_df = join_case_assignment_df.select(
            case_assignment.AssignmentId.cast("string").alias("AssignmentId"),
            lit("qca").alias("SourceSystem"),
            portfolio.portfolio_id_qca.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).cast("String").alias("BatchId"),
            case_assignment.AssignmentKey.cast("String").alias("AssignmentKey"),
            case_assignment.case_code.cast("Int").alias("CaseCode"),
            case_assignment.case_account_for_search.cast("String").alias("ExposureKey"),
            case_assignment.cust_code.cast("String").alias("PartyKey"),
            join_case_assignment_df.cust_cif.cast("String").alias("CDI"),
            join_case_assignment_df.cust_afm.cast("String").alias(
                "TaxIdentificationNumber"
            ),
            lit(None).cast("String").alias("AgreementKey"),
            lit(None).cast("String").alias("InternalAssignmentStartDate"),
            lit(None).cast("String").alias("ExternalAssignmentStartDate"),
            lit(None).cast("timestamp").alias("AssignmentRecallDate"),
            join_case_assignment_df.Channel.cast("String").alias("ChannelCode"),
            join_case_assignment_df.Channel.cast("String").alias("Channel"),
            lit(None).cast("String").alias("ExternalCompanyAgency"),
            case_assignment.externalcompanyassignment.cast("String").alias(
                "ExternalCompanyPartyKey"
            ),
            case_assignment.externalcompanyname.cast("String").alias(
                "ExternalCompanyFullName"
            ),
            lit("QCAB004").cast("String").alias("AssignmentTypeCode"),
            lit("Channel Assignment").cast("String").alias("AssignmentType"),
            join_case_assignment_df.Team.cast("String").alias("TeamCode"),
            join_case_assignment_df.Team.cast("String").alias("Team"),
            dim_collector.col_id.cast("String").alias("AgentPartyKey"),
            dim_collector.Agent_Full_Name.cast("String").alias("AgentFullName"),
            dim_collector.col_login.cast("String").alias("AgentLoginName"),
            lit(None).cast("String").alias("Supervisor"),
            lit(None).cast("String").alias("AssistantManager"),
            lit(None).cast("String").alias("Manager"),
            lit(None).cast("String").alias("Head"),
            lit(None).cast("Int").alias("AssignmentBucketCode"),
            lit(None).cast("String").alias("AssignmentBucket"),
            case_assignment.AssignmentProcess.cast("String").alias("AssignmentProcess"),
            dim_packet_hier3.PACKH3_DESC.cast("String").alias("ProcessType"),
            dim_packet.pack_id.cast("Int").alias("PacketCode"),
            dim_packet.pack_name.cast("String").alias("Packet"),
            lit(None).cast("timestamp").alias("PacketPreviousEndDate"),
            dim_packet.pack_assign_date.cast("timestamp").alias("PacketAssignmentDate"),
            fs_cases_snap.pacc_code.cast("Int").alias("PaccCode"),
            dim_queue.que_id.cast("Int").alias("QueCode"),
            dim_queue.que_desc.cast("String").alias("Que"),
            at_customer_extra.CUEX_GENERIC_TEXT_16.cast("String").alias("Segment"),
            at_customer_extra.CUEX_GENERIC_TEXT_20.cast("String").alias("ExtraSegment"),
            lit(None).cast("Int").alias("StreamTypeCode"),
            lit(None).cast("String").alias("StreamType"),
            lit(None).cast("Int").alias("StreamStatusCode"),
            lit(None).cast("String").alias("StreamStatus"),
            lit(None).cast("Int").alias("StreamAssignedStatusCode"),
            lit(None).cast("String").alias("StreamAssignedStatus"),
            lit(None).cast("double").alias("OutstandingAmount"),
            lit(None).cast("double").alias("DebtAmount"),
            lit(None).cast("String").alias("CampaignName"),
            case_assignment.businessdate.alias("BusinessDate"),
            join_case_assignment_df.ForcedOnboarding.cast("Boolean"),
            join_case_assignment_df.HolisticFlag.cast("Boolean"),
            join_case_assignment_df.ForcedOnboardReason.cast("String"),
        )

        # fs_assignment_snap_leaf_df
        fs_assignment_snap_leaf_df = join_fs_assignment_snap_leaf_df.select(
            join_fs_assignment_snap_leaf_df.AssignmentId.cast("string").alias("AssignmentId"),
            lit("qca").alias("SourceSystem"),
            portfolio.portfolio_id_qca.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias("IngestionTimestamp"),
            lit(None).cast("String").alias("BatchId"),
            join_fs_assignment_snap_leaf_df.AssignmentKey.cast("String").alias("AssignmentKey"),
            fs_assignment_snap_leaf.case_code.cast("Int").alias("CaseCode"),
            join_fs_assignment_snap_leaf_df.case_account_for_search.cast("String").alias("ExposureKey"),
            fs_assignment_snap_leaf.cust_code.cast("String").alias("PartyKey"),
            dim_customer_static.cust_cif.cast("String").alias("CDI"),
            dim_customer_static.cust_afm.cast("String").alias("TaxIdentificationNumber"),
            lit(None).cast("String").alias("AgreementKey"),
            join_fs_assignment_snap_leaf_df.concatenated_cr_date.cast("String").alias("InternalAssignmentStartDate"),
            lit(None).cast("String").alias("ExternalAssignmentStartDate"),
            lit(None).cast("timestamp").alias("AssignmentRecallDate"),
            lit(None).cast("String").alias("ChannelCode"),
            lit(None).cast("String").alias("Channel"),
            lit(None).cast("String").alias("ExternalCompanyAgency"),
            join_fs_assignment_snap_leaf_df.extc_code_primary.cast("String").alias("ExternalCompanyPartyKey"),
            dim_external_company.DEC_EXTC_NAME.cast("String").alias(
                "ExternalCompanyFullName"
            ),
            lit("QCAB001").cast("String").alias("AssignmentTypeCode"),
            lit("Collection Assignment").cast("String").alias("AssignmentType"),
            lit(None).cast("String").alias("TeamCode"),
            lit(None).cast("String").alias("Team"),
            fs_assignment_snap_leaf.ass_col_id_ass.cast("String").alias(
                "AgentPartyKey"
            ),
            dim_collector.Agent_Full_Name.cast("String").alias(
                "AgentFullName"
            ),
            dim_collector.col_login.cast("String").alias(
                "AgentLoginName"
            ),
            lit(None).cast("String").alias("Supervisor"),
            lit(None).cast("String").alias("AssistantManager"),
            lit(None).cast("String").alias("Manager"),
            lit(None).cast("String").alias("Head"),
            join_fs_assignment_snap_leaf_df.bck_ass_id.cast("Int").alias("AssignmentBucketCode"),
            join_fs_assignment_snap_leaf_df.bck_desc.cast("String").alias("AssignmentBucket"),
            lit(None).cast("String").alias("AssignmentProcess"),
            dim_packet_hier3.PACKH3_DESC.cast("String").alias("ProcessType"),
            fs_assignment_snap_leaf.pack_id.cast("Int").alias("PacketCode"),
            join_fs_assignment_snap_leaf_df.pack_name.cast("String").alias("Packet"),
            join_fs_assignment_snap_leaf_df.pacc_dateback.cast("timestamp").alias("PacketPreviousEndDate"),
            join_fs_assignment_snap_leaf_df.pack_assign_date.cast("timestamp").alias("PacketAssignmentDate"),
            fs_assignment_snap_leaf.pacc_code.cast("Int").alias("PaccCode"),
            join_fs_assignment_snap_leaf_df.caqu_que_code.cast("Int").alias("QueCode"),
            join_fs_assignment_snap_leaf_df.caqu_que_desc.cast("String").alias("Que"),
            at_customer_extra.CUEX_GENERIC_TEXT_16.cast("String").alias("Segment"),
            at_customer_extra.CUEX_GENERIC_TEXT_20.cast("String").alias("ExtraSegment"),
            lit(None).cast("Int").alias("StreamTypeCode"),
            lit(None).cast("String").alias("StreamType"),
            lit(None).cast("Int").alias("StreamStatusCode"),
            lit(None).cast("String").alias("StreamStatus"),
            lit(None).cast("Int").alias("StreamAssignedStatusCode"),
            lit(None).cast("String").alias("StreamAssignedStatus"),
            join_fs_assignment_snap_leaf_df.fsas_out_amt_ass.cast("double").alias("OutstandingAmount"),
            join_fs_assignment_snap_leaf_df.fsas_dbt_amt.cast("double").alias("DebtAmount"),
            lit(None).cast("String").alias("CampaignName"),
            fs_assignment_snap_leaf.businessdate.alias("BusinessDate"),
            join_fs_assignment_snap_leaf_df.ForcedOnboarding.cast("Boolean"),
            join_fs_assignment_snap_leaf_df.HolisticFlag.cast("Boolean"),
            join_fs_assignment_snap_leaf_df.ForcedOnboardReason.cast("String"),
        )

        # fs_case_stream_assignments_snap_leaf
        fs_case_stream_assignments_snap_leaf_df = join_fs_case_stream_assignments_snap_leaf_df.select(
            join_fs_case_stream_assignments_snap_leaf_df.AssignmentId.cast("string").alias("AssignmentId"),
            lit("qca").alias("SourceSystem"),
            portfolio.portfolio_id_qca.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias("IngestionTimestamp"),
            lit(None).cast("String").alias("BatchId"),
            join_fs_case_stream_assignments_snap_leaf_df.AssignmentKey.cast("String").alias("AssignmentKey"),
            join_fs_case_stream_assignments_snap_leaf_df.fs_case_stream_assignments_snap_leaf_case_code.cast("Int").alias("CaseCode"),
            join_fs_case_stream_assignments_snap_leaf_df.case_account_for_search.cast("String").alias("ExposureKey"),
            fs_case_stream_assignments_snap_leaf.cust_code.cast("String").alias("PartyKey"),
            dim_customer_static.cust_cif.cast("String").alias("CDI"),
            dim_customer_static.cust_afm.cast("String").alias("TaxIdentificationNumber"),
            lit(None).cast("String").alias("AgreementKey"),
            join_fs_case_stream_assignments_snap_leaf_df.concatenated_cr_date.cast("String").alias("InternalAssignmentStartDate"),
            lit(None).cast("String").alias("ExternalAssignmentStartDate"),
            lit(None).cast("timestamp").alias("AssignmentRecallDate"),
            lit(None).cast("String").alias("ChannelCode"),
            lit(None).cast("String").alias("Channel"),
            lit(None).cast("String").alias("ExternalCompanyAgency"),
            dim_external_company.DEC_extc_id.cast("String").alias(
                "ExternalCompanyPartyKey"
            ),
            dim_external_company.DEC_EXTC_NAME.cast("String").alias(
                "ExternalCompanyFullName"
            ),
            lit("QCAB002").cast("String").alias("AssignmentTypeCode"),
            lit("Legal Action Assignment").cast("String").alias("AssignmentType"),
            lit(None).cast("String").alias("TeamCode"),
            lit(None).cast("String").alias("Team"),
            lit(None).cast("String").alias("AgentPartyKey"),
            lit(None).cast("String").alias("AgentFullName"),
            lit(None).cast("String").alias("Supervisor"),
            lit(None).cast("String").alias("AssistantManager"),
            lit(None).cast("String").alias("Manager"),
            lit(None).cast("String").alias("Head"),
            lit(None).cast("String").alias("AgentLoginName"),
            join_fs_case_stream_assignments_snap_leaf_df.pacc_bucket.cast("Int").alias("AssignmentBucketCode"),
            join_fs_case_stream_assignments_snap_leaf_df.bck_desc.cast("String").alias("AssignmentBucket"),
            lit(None).cast("String").alias("AssignmentProcess"),
            dim_packet_hier3.PACKH3_DESC.cast("String").alias("ProcessType"),
            join_fs_case_stream_assignments_snap_leaf_df.pack_code.cast("Int").alias("PacketCode"),
            join_fs_case_stream_assignments_snap_leaf_df.pack_name.cast("String").alias("Packet"),
            join_fs_case_stream_assignments_snap_leaf_df.pacc_dateback.cast("timestamp").alias("PacketPreviousEndDate"),
            fs_case_stream_assignments_snap_leaf.pack_assign_date.cast("timestamp").alias("PacketAssignmentDate"),
            fs_case_stream_assignments_snap_leaf.pacc_code.cast("Int").alias("PaccCode"),
            join_fs_case_stream_assignments_snap_leaf_df.caqu_que_code.cast("Int").alias("QueCode"),
            join_fs_case_stream_assignments_snap_leaf_df.caqu_que_desc.cast("String").alias("Que"),
            lit(None).cast("String").alias("Segment"),
            lit(None).cast("String").alias("ExtraSegment"),
            join_fs_case_stream_assignments_snap_leaf_df.fs_case_stream_assignments_snap_leaf_strm_id.cast("Int").alias("StreamTypeCode"),
            join_fs_case_stream_assignments_snap_leaf_df.strm_desc.cast("String").alias("StreamType"),
            join_fs_case_stream_assignments_snap_leaf_df.strm_sta_id.cast("Int").alias("StreamStatusCode"),
            join_fs_case_stream_assignments_snap_leaf_df.strm_stat_desc.cast("String").alias("StreamStatus"),
            join_fs_case_stream_assignments_snap_leaf_df.strm_sta_ass_id.cast("Int").alias("StreamAssignedStatusCode"),
            join_fs_case_stream_assignments_snap_leaf_df.dim_stream_status_alias2_strm_stat_desc.cast("String").alias("StreamAssignedStatus"),
            join_fs_case_stream_assignments_snap_leaf_df.pacc_amount.cast("double").alias("OutstandingAmount"),
            join_fs_case_stream_assignments_snap_leaf_df.pacc_debt_amount.cast("double").alias("DebtAmount"),
            lit(None).cast("String").alias("CampaignName"),
            fs_case_stream_assignments_snap_leaf.businessdate.alias("BusinessDate"),
            join_fs_case_stream_assignments_snap_leaf_df.ForcedOnboarding.cast("Boolean"),
            join_fs_case_stream_assignments_snap_leaf_df.HolisticFlag.cast("Boolean"),
            join_fs_case_stream_assignments_snap_leaf_df.ForcedOnboardReason.cast("String"),
        )

        # fs_customer_stream_assignments_snap_leaf
        fs_customer_stream_assignments_snap_leaf = join_fs_customer_stream_assignments_snap_leaf_df.select(
            join_fs_customer_stream_assignments_snap_leaf_df.AssignmentId.cast(
                "string"
            ).alias("AssignmentId"),
            lit("qca").alias("SourceSystem"),
            coalesce(portfolio.portfolio_id_qca, lit(2))
            .cast("string")
            .alias("PortfolioKey"),
            coalesce(portfolio.PortfolioName, lit("Core Retail"))
            .cast("string")
            .alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).cast("String").alias("BatchId"),
            join_fs_customer_stream_assignments_snap_leaf_df.AssignmentKey.cast(
                "String"
            ).alias("AssignmentKey"),
            fs_customer_stream_assignments_snap_leaf.case_code.cast("Int").alias(
                "CaseCode"
            ),
            lit(None).cast("String").alias("ExposureKey"),
            fs_customer_stream_assignments_snap_leaf.cust_code.cast("String").alias(
                "PartyKey"
            ),
            dim_customer_static.cust_cif.cast("String").alias("CDI"),
            dim_customer_static.cust_afm.cast("String").alias(
                "TaxIdentificationNumber"
            ),
            lit(None).cast("String").alias("AgreementKey"),
            join_fs_customer_stream_assignments_snap_leaf_df.concatenated_cr_date.cast(
                "String"
            ).alias("InternalAssignmentStartDate"),
            lit(None).cast("String").alias("ExternalAssignmentStartDate"),
            lit(None).cast("timestamp").alias("AssignmentRecallDate"),
            lit(None).cast("String").alias("ChannelCode"),
            lit(None).cast("String").alias("Channel"),
            lit(None).cast("String").alias("ExternalCompanyAgency"),
            join_fs_customer_stream_assignments_snap_leaf_df.extc_code.cast(
                "String"
            ).alias("ExternalCompanyPartyKey"),
            dim_external_company.DEC_EXTC_NAME.cast("String").alias(
                "ExternalCompanyFullName"
            ),
            lit("QCAB003").cast("String").alias("AssignmentTypeCode"),
            lit("Real Estate Research Assignment")
            .cast("String")
            .alias("AssignmentType"),
            lit(None).cast("String").alias("TeamCode"),
            lit(None).cast("String").alias("Team"),
            lit(None)
            .cast("String")
            .alias("AgentPartyKey"),  # renamed previous AssignmentAgentPartyKey
            lit(None).cast("String").alias("AgentFullName"),  # change
            lit(None).cast("String").alias("AgentLoginName"),
            lit(None).cast("String").alias("Supervisor"),
            lit(None).cast("String").alias("AssistantManager"),
            lit(None).cast("String").alias("Manager"),
            lit(None).cast("String").alias("Head"),
            lit(None).cast("Int").alias("AssignmentBucketCode"),
            lit(None).cast("String").alias("AssignmentBucket"),
            lit(None).cast("String").alias("AssignmentProcess"),
            lit(None).cast("String").alias("ProcessType"),
            join_fs_customer_stream_assignments_snap_leaf_df.pack_code.cast(
                "Int"
            ).alias("PacketCode"),
            join_fs_customer_stream_assignments_snap_leaf_df.pack_name.cast(
                "String"
            ).alias("Packet"),
            lit(None).cast("timestamp").alias("PacketPreviousEndDate"),
            lit(None).cast("timestamp").alias("PacketAssignmentDate"),
            fs_customer_stream_assignments_snap_leaf.pacc_code.cast("Int").alias(
                "PaccCode"
            ),
            lit(None).cast("Int").alias("QueCode"),
            lit(None).cast("String").alias("Que"),
            lit(None).cast("String").alias("Segment"),
            lit(None).cast("String").alias("ExtraSegment"),
            lit(None).cast("Int").alias("StreamTypeCode"),
            lit(None).cast("String").alias("StreamType"),
            lit(None).cast("Int").alias("StreamStatusCode"),
            lit(None).cast("String").alias("StreamStatus"),
            lit(None).cast("Int").alias("StreamAssignedStatusCode"),
            lit(None).cast("String").alias("StreamAssignedStatus"),
            join_fs_customer_stream_assignments_snap_leaf_df.pacc_outstanding_amt_lcy.cast(
                "double"
            ).alias(
                "OutstandingAmount"
            ),
            lit(None).cast("double").alias("DebtAmount"),
            lit(None).cast("String").alias("CampaignName"),
            fs_customer_stream_assignments_snap_leaf.businessdate.alias("BusinessDate"),
            lit(None).cast("Boolean").alias("ForcedOnboarding"),
            lit(None).cast("Boolean").alias("HolisticFlag"),
            lit(None).cast("String").alias("ForcedOnboardReason"),
        )

        # ================== Selection of Final Columns ==================
        # ================== union all ==================

        result_df_qca = (
            case_assignment_df.union(fs_assignment_snap_leaf_df)
            .union(fs_case_stream_assignments_snap_leaf_df)
            .union(fs_customer_stream_assignments_snap_leaf)
        )

        # ================== Append ==================
        print("Saving to Delta table")
        curated_zone = "`qa-cdh`.primaryactive"
        curated_table_path = f"{curated_zone}.Assignment"
        result_df_qca.write.format("delta").mode(
            "append"
        ).saveAsTable(curated_table_path)
        print("Saved to Delta table")



In [0]:
%sql
SELECT DISTINCT BusinessDate
FROM `qa-cdh`.primaryactive.Assignment
WHERE PortfolioKey = 61
ORDER BY BusinessDate DESC

##### Run qca Assignment function and save results into primaryactive.Assignment table

In [0]:
spark = spark
catalog_landed = "`p-cdh`.landed"
run_date = '2026-10-05'
env_prefix = 'p'
load_curated_assignment_qca(
    spark = spark,
    catalog_landed = catalog_landed,
    run_date = run_date,
    env_prefix = env_prefix
)

##### Investigate & Resolve Assignment duplicates

In [0]:
%python
from assignment_internal import AssignmentInternal
spark = spark
catalog_landed = "`p-cdh`.landed"
run_date = '2026-10-05'
env_prefix = 'p'
assignment_internal = AssignmentInternal(spark, catalog_landed, run_date, env_prefix)
case_assignment, s1, s2, s3, s4, s5, s6 = assignment_internal.loadCuratedAssignmentQca()
case_assignment.createOrReplaceTempView('case_assignment')
s6.createOrReplaceTempView('s6')
s5.createOrReplaceTempView('s5')
s4.createOrReplaceTempView('s4')
s3.createOrReplaceTempView('s3')
s2.createOrReplaceTempView('s2')
s1.createOrReplaceTempView('s1')

##### Correct Duplicates and AgentFullName entry

##### 1. BusinessDate = '2026-10-05'

In [0]:
%sql
SELECT AssignmentProcess, ExternalCompanyFullName, AgentFullName, *
FROM `qa-cdh`.primaryactive.Assignment
WHERE BusinessDate = '2026-10-05'
AND SourceSystem = 'qca'
AND ExposureKey = '989007027361800'
AND CaseCode = '100122'
ORDER BY AssignmentKey
LIMIT 100

In [0]:
%sql
WITH RawAssignments AS (
SELECT *
FROM `qa-cdh`.primaryactive.Assignment
WHERE BusinessDate = '2026-10-05'
AND SourceSystem = 'qca'
AND PortfolioName = 'Athena Portfolio'
),

withDuplAssignments AS (
SELECT AssignmentKey AS DuplicatedAssignmentKey, CountRows
FROM (
SELECT AssignmentKey, COUNT(*) AS CountRows
FROM RawAssignments
GROUP BY AssignmentKey
)
WHERE CountRows = 2
),

withDuplRawAssignments AS (
SELECT x.*, y.CountRows
FROM RawAssignments AS x
INNER JOIN withDuplAssignments AS y
ON x.AssignmentKey = y.DuplicatedAssignmentKey
ORDER BY x.AssignmentKey
)

-- Initial rows in the table: 38,429
-- Before dedup: 38,394 
-- After dedup: 19,197
-- We want to keep these AFTER CORRECTION: 19,232
/*
SELECT COUNT(*) -- AssignmentProcess, ExternalCompanyFullName, AgentFullName, *
FROM withDuplRawAssignments
WHERE AssignmentProcess = 'Passive'
AND ExternalCompanyFullName = 'N/a'
*/

SELECT COUNT(*) -- AssignmentProcess, ExternalCompanyFullName, AgentFullName, *
FROM withDuplRawAssignments
WHERE AssignmentProcess = 'N/A'
AND ExternalCompanyFullName IS NULL

/*
1. AssignmentProcess:
Passive
N/A

2. ExternalCompanyFullName:
null
N/a

3. AgentFullName:
null
*/

In [0]:
%sql
-- Query 1.1: BusinessDate = '2026-10-05'
WITH RawAssignments AS (
SELECT *
FROM `qa-cdh`.primaryactive.Assignment
WHERE BusinessDate = '2026-10-05'
AND SourceSystem = 'qca'
AND PortfolioName = 'Athena Portfolio'
),

withDuplAssignments AS (
SELECT AssignmentKey
FROM (
SELECT AssignmentKey, COUNT(*) AS CountRows
FROM RawAssignments
GROUP BY AssignmentKey
)
WHERE CountRows = 2
)

DELETE FROM `qa-cdh`.primaryactive.Assignment
WHERE BusinessDate = '2026-10-05'
AND SourceSystem = 'qca'
AND PortfolioName = 'Athena Portfolio'
AND AssignmentProcess = 'N/A'
AND ExternalCompanyFullName IS NULL
AND AssignmentKey IN (SELECT AssignmentKey FROM withDuplAssignments)

/*
-- Number of affected/to be deleted rows: 19197 for BusinessDate = '2026-10-05'
-- Μετα την διόρθωση το παρακάτω query πρέπει να επιστρέψει 19232 rows
*/

In [0]:
%sql
-- Query 1.2
-- Update AgentFullName for duplicate AssignmentKeys
-- BusinessDate = '2026-10-05'

WITH RawAssignments AS (
    SELECT *
    FROM `qa-cdh`.primaryactive.Assignment
    WHERE BusinessDate = '2026-10-05'
      AND SourceSystem = 'qca'
      AND PortfolioName = 'Athena Portfolio'
)

UPDATE `qa-cdh`.primaryactive.Assignment
SET AgentFullName = 'N/a'
WHERE BusinessDate = '2026-10-05'
  AND SourceSystem = 'qca'
  AND PortfolioName = 'Athena Portfolio'
  AND AssignmentProcess = 'Passive'
  AND ExternalCompanyFullName = 'N/a'
  AND AgentFullName IS NULL;

In [0]:
%sql
SELECT AssignmentProcess, ExternalCompanyFullName, AgentFullName, *
FROM `qa-cdh`.primaryactive.Assignment
WHERE BusinessDate = '2026-10-05'
AND SourceSystem = 'qca'
AND ExposureKey = '989007027361800'
AND CaseCode = '100122'
ORDER BY AssignmentKey
LIMIT 100

In [0]:
%sql
SELECT COUNT(*), COUNT(DISTINCT AssignmentKey) -- 19,232 both
FROM `qa-cdh`.primaryactive.Assignment
WHERE BusinessDate = '2026-10-05'
AND SourceSystem = 'qca'
AND PortfolioName = 'Athena Portfolio'

##### 2. BusinessDate = '2026-10-06'

In [0]:
%sql
SELECT AssignmentProcess, ExternalCompanyFullName, AgentFullName, *
FROM `p-cdh`.primaryactive.Assignment
WHERE BusinessDate = '2026-10-06'
AND SourceSystem = 'qca'
AND ExposureKey = '989007027361800'
AND CaseCode = '100122'
ORDER BY AssignmentKey
LIMIT 100

In [0]:
%sql
WITH RawAssignments AS (
SELECT *
FROM `qa-cdh`.primaryactive.Assignment
WHERE BusinessDate = '2026-10-06'
AND SourceSystem = 'qca'
AND PortfolioName = 'Athena Portfolio'
),

withDuplAssignments AS (
SELECT AssignmentKey AS DuplicatedAssignmentKey, CountRows
FROM (
SELECT AssignmentKey, COUNT(*) AS CountRows
FROM RawAssignments
GROUP BY AssignmentKey
)
WHERE CountRows = 2
),

withDuplRawAssignments AS (
SELECT x.*, y.CountRows
FROM RawAssignments AS x
INNER JOIN withDuplAssignments AS y
ON x.AssignmentKey = y.DuplicatedAssignmentKey
ORDER BY x.AssignmentKey
)

-- Initial rows in the table: 38,388
-- Distinct AssignmentKeys: 19,194
-- We want to keep these AFTER CORRECTION: 19,232

SELECT COUNT(*) -- 19,194
FROM withDuplRawAssignments
WHERE AssignmentProcess = 'Passive'
AND ExternalCompanyFullName = 'N/a'

In [0]:
%sql
-- Query 2.1: BusinessDate = '2026-10-06'
WITH RawAssignments AS (
SELECT *
FROM `qa-cdh`.primaryactive.Assignment
WHERE BusinessDate = '2026-10-06'
AND SourceSystem = 'qca'
AND PortfolioName = 'Athena Portfolio'
),

withDuplAssignments AS (
SELECT AssignmentKey
FROM (
SELECT AssignmentKey, COUNT(*) AS CountRows
FROM RawAssignments
GROUP BY AssignmentKey
)
WHERE CountRows = 2
)

DELETE FROM `qa-cdh`.primaryactive.Assignment
WHERE BusinessDate = '2026-10-06'
AND SourceSystem = 'qca'
AND PortfolioName = 'Athena Portfolio'
AND AssignmentProcess = 'N/A'
AND ExternalCompanyFullName IS NULL
AND AssignmentKey IN (SELECT AssignmentKey FROM withDuplAssignments)

/*
-- Number of affected/to be deleted rows: 19197 for BusinessDate = '2026-10-06'
-- Μετα την διόρθωση το παρακάτω query πρέπει να επιστρέψει 19,194 rows
*/

In [0]:
%sql
-- Query 2.2
-- Update AgentFullName for duplicate AssignmentKeys
-- BusinessDate = '2026-10-06'

WITH RawAssignments AS (
    SELECT *
    FROM `qa-cdh`.primaryactive.Assignment
    WHERE BusinessDate = '2026-10-06'
      AND SourceSystem = 'qca'
      AND PortfolioName = 'Athena Portfolio'
)

UPDATE `qa-cdh`.primaryactive.Assignment
SET AgentFullName = 'N/a'
WHERE BusinessDate = '2026-10-06'
  AND SourceSystem = 'qca'
  AND PortfolioName = 'Athena Portfolio'
  AND AssignmentProcess = 'Passive'
  AND ExternalCompanyFullName = 'N/a'
  AND AgentFullName IS NULL;

In [0]:
%sql
SELECT AssignmentProcess, ExternalCompanyFullName, AgentFullName, *
FROM `qa-cdh`.primaryactive.Assignment
WHERE BusinessDate = '2026-10-06'
AND SourceSystem = 'qca'
AND ExposureKey = '989007027361800'
AND CaseCode = '100122'
ORDER BY AssignmentKey
LIMIT 100

In [0]:
%sql
SELECT COUNT(*), COUNT(DISTINCT AssignmentKey) -- 19,194 both
FROM `qa-cdh`.primaryactive.Assignment
WHERE BusinessDate = '2026-10-06'
AND SourceSystem = 'qca'
AND PortfolioName = 'Athena Portfolio'